# Chicago Crime - Notebook Predictions

Ce notebook reprend la partie modelisation du notebook `chicago_crime_ML_v3.ipynb` et la separe dans un parcours prediction clair.

Pipeline :
1. K-Means geographique pour identifier une zone.
2. Random Forest pour predire la categorie de crime.
3. XGBoost pour predire la probabilite d'arrestation en utilisant la categorie predite.

Les artefacts sont sauvegardes dans `artifacts/models/`.


## Analyse du notebook existant

Le notebook source avait une bonne idee de pipeline en deux etapes : predire d'abord le type de crime, puis l'arrestation. Les points gardes ici :

- source de donnees Chicago Data Portal avec filtre depuis 2020 ;
- regroupement des lieux en categories metier ;
- top crimes + classe `OTHER` ;
- K-Means comme feature spatiale ;
- Random Forest pour le multiclass ;
- XGBoost pour le binaire desequilibre.

Amelioration apportee : le cluster K-Means est predit pour toutes les lignes nettoyees, pas seulement pour l'echantillon utilise lors de l'apprentissage du clustering.


## 0. Imports et configuration


In [ ]:
from pathlib import Path
from io import StringIO
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

RANDOM_STATE = 42
START_YEAR = 2020
API_LIMIT = 200_000
N_CLUSTERS = 5
RUN_HYPERPARAM_SEARCH = False

DATA_URL = 'https://data.cityofchicago.org/resource/ijzp-q8t2.csv'

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'artifacts' / 'models'
PRED_DIR = PROJECT_ROOT / 'artifacts' / 'predictions'
for path in [DATA_DIR, MODEL_DIR, PRED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_CANDIDATES = [
    DATA_DIR / 'chicago_crimes.csv',
    DATA_DIR / 'chicago_crimes_sample.csv',
    PROJECT_ROOT / 'chicago_crimes.csv',
]

print(f'Project root : {PROJECT_ROOT}')
print(f'Model outputs: {MODEL_DIR}')


## 1. Chargement des donnees


In [ ]:
def load_chicago_crimes(force_api=False):
    """Load Chicago crimes from a local CSV if available, otherwise from the public API."""
    if not force_api:
        for path in LOCAL_DATA_CANDIDATES:
            if path.exists():
                print(f'Chargement local : {path}')
                return pd.read_csv(path, low_memory=False)

    params = {
        '$limit': API_LIMIT,
        '$order': 'date DESC',
        '$where': f'year >= {START_YEAR}',
    }
    print('Chargement depuis l API Chicago...')
    response = requests.get(DATA_URL, params=params, timeout=120)
    response.raise_for_status()
    df_loaded = pd.read_csv(StringIO(response.text), low_memory=False)

    cache_path = DATA_DIR / 'chicago_crimes_sample.csv'
    df_loaded.to_csv(cache_path, index=False)
    print(f'Cache ecrit : {cache_path}')
    return df_loaded


df_raw = load_chicago_crimes()
print(f'Shape brute : {df_raw.shape}')
df_raw.head(3)


## 2. Nettoyage et features


In [ ]:
def to_binary(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int)
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y']).astype(int)


LOCATION_GROUP_MAPPING = {
    'Street': 0,
    'Residence': 1,
    'Commercial': 2,
    'Transport': 3,
    'Public': 4,
    'Other': 5,
}


def location_group_label(location):
    if pd.isna(location):
        return 'Other'
    loc = str(location).upper()
    if any(word in loc for word in ['STREET', 'ALLEY', 'SIDEWALK', 'PARKING']):
        return 'Street'
    if any(word in loc for word in ['RESIDENCE', 'APARTMENT', 'HOUSE']):
        return 'Residence'
    if any(word in loc for word in ['STORE', 'RESTAURANT', 'BANK', 'RETAIL', 'GAS']):
        return 'Commercial'
    if any(word in loc for word in ['CTA', 'BUS', 'TRAIN', 'VEHICLE']):
        return 'Transport'
    if any(word in loc for word in ['PARK', 'SCHOOL', 'CHURCH', 'HOSPITAL']):
        return 'Public'
    return 'Other'


def prepare_model_data(df_input):
    df = df_input.copy()
    df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date_parsed', 'latitude', 'longitude', 'primary_type', 'arrest'])
    df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]

    df['hour'] = df['date_parsed'].dt.hour
    df['day_of_week'] = df['date_parsed'].dt.dayofweek
    df['month'] = df['date_parsed'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['arrest'] = to_binary(df['arrest'])
    df['domestic'] = to_binary(df['domestic'])

    df['location_group_label'] = df['location_description'].apply(location_group_label)
    df['location_group'] = df['location_group_label'].map(LOCATION_GROUP_MAPPING).fillna(5).astype(int)

    top10_crimes = df['primary_type'].value_counts().head(10).index.tolist()
    df['crime_cat'] = df['primary_type'].where(df['primary_type'].isin(top10_crimes), 'OTHER')
    return df, top10_crimes


df, top10_crimes = prepare_model_data(df_raw)
print(f'Lignes nettoyees : {len(df):,}')
print(f'Top 10 crimes    : {top10_crimes}')
print(f'Taux arrestation : {df["arrest"].mean():.1%}')
df[['date_parsed', 'primary_type', 'crime_cat', 'arrest', 'latitude', 'longitude', 'location_group_label']].head()


## 3. Step 0 - Clustering geographique


In [ ]:
coords = df[['latitude', 'longitude']].copy()
coords_sample = coords.sample(min(50_000, len(coords)), random_state=RANDOM_STATE)

scaler_kmeans = StandardScaler()
X_spatial_sample = scaler_kmeans.fit_transform(coords_sample)

inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_spatial_sample)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, 11), inertias, marker='o', linewidth=2, color='#4c78a8')
ax.axvline(N_CLUSTERS, color='#e15759', linestyle='--', label=f'k={N_CLUSTERS}')
ax.set_title('Choix du nombre de clusters - methode elbow', fontweight='bold')
ax.set_xlabel('Nombre de clusters')
ax.set_ylabel('Inertie')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(PRED_DIR / '01_kmeans_elbow.png', bbox_inches='tight')
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_spatial_sample)

X_spatial_all = scaler_kmeans.transform(coords)
df['cluster'] = kmeans.predict(X_spatial_all)

cluster_profile = (
    df.groupby('cluster')
    .agg(
        volume=('cluster', 'size'),
        arrest_rate=('arrest', 'mean'),
        latitude=('latitude', 'mean'),
        longitude=('longitude', 'mean'),
    )
    .sort_index()
)
cluster_profile.style.format({
    'volume': '{:,.0f}',
    'arrest_rate': '{:.1%}',
    'latitude': '{:.4f}',
    'longitude': '{:.4f}',
})


In [ ]:
geo_sample = df.sample(min(25_000, len(df)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(8, 8))
for cluster_id, group in geo_sample.groupby('cluster'):
    ax.scatter(group['longitude'], group['latitude'], s=3, alpha=0.4, label=f'Cluster {cluster_id}')
centroids = scaler_kmeans.inverse_transform(kmeans.cluster_centers_)
ax.scatter(centroids[:, 1], centroids[:, 0], c='black', marker='X', s=120, label='Centroïdes')
ax.set_title('Clusters geographiques Chicago', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=3, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(PRED_DIR / '02_kmeans_clusters.png', bbox_inches='tight')
plt.show()


## 4. Step 1 - Prediction du type de crime


In [ ]:
le_crime = LabelEncoder()
df['crime_encoded'] = le_crime.fit_transform(df['crime_cat'])

FEAT_STEP1 = [
    'latitude', 'longitude', 'hour', 'day_of_week', 'month',
    'is_weekend', 'cluster', 'location_group',
]
TARGET_STEP1 = 'crime_encoded'

model_df_s1 = df[FEAT_STEP1 + [TARGET_STEP1]].dropna()
X1 = model_df_s1[FEAT_STEP1]
y1 = model_df_s1[TARGET_STEP1]

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=RANDOM_STATE, stratify=y1
)

print(f'Dataset Step 1 : {len(model_df_s1):,} lignes')
print(f'Train/Test     : {len(X1_train):,} / {len(X1_test):,}')
print(f'Classes        : {le_crime.classes_.tolist()}')


In [ ]:
rf_crime = RandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_crime.fit(X1_train, y1_train)

y1_pred = rf_crime.predict(X1_test)
acc_s1 = accuracy_score(y1_test, y1_pred)
f1_s1 = f1_score(y1_test, y1_pred, average='weighted')

print('STEP 1 - Random Forest')
print(f'Accuracy : {acc_s1:.3f}')
print(f'F1 score : {f1_s1:.3f}')
print()
print(classification_report(y1_test, y1_pred, target_names=le_crime.classes_))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

importances_s1 = pd.Series(rf_crime.feature_importances_, index=FEAT_STEP1).sort_values()
axes[0].barh(importances_s1.index, importances_s1.values, color='#4c78a8', edgecolor='white')
axes[0].set_title('Feature importance - crime type', fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)

cm1 = confusion_matrix(y1_test, y1_pred, normalize='true')
sns.heatmap(
    cm1, cmap='Blues', ax=axes[1], xticklabels=le_crime.classes_, yticklabels=le_crime.classes_,
    linewidths=0.2, cbar_kws={'shrink': 0.8}
)
axes[1].set_title('Matrice de confusion normalisee', fontweight='bold')
axes[1].set_xlabel('Predit')
axes[1].set_ylabel('Reel')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig(PRED_DIR / '03_step1_random_forest.png', bbox_inches='tight')
plt.show()


## 5. Step 2 - Prediction de l arrestation


In [ ]:
df['crime_predicted'] = rf_crime.predict(df[FEAT_STEP1].fillna(0))

FEAT_STEP2 = [
    'latitude', 'longitude', 'hour', 'day_of_week', 'month',
    'is_weekend', 'domestic', 'cluster', 'location_group', 'crime_predicted',
]
TARGET_STEP2 = 'arrest'

model_df_s2 = df[FEAT_STEP2 + [TARGET_STEP2]].dropna()
X2 = model_df_s2[FEAT_STEP2]
y2 = model_df_s2[TARGET_STEP2]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=RANDOM_STATE, stratify=y2
)

scale_pos_weight = (y2_train == 0).sum() / max((y2_train == 1).sum(), 1)
print(f'Dataset Step 2    : {len(model_df_s2):,} lignes')
print(f'Train/Test        : {len(X2_train):,} / {len(X2_test):,}')
print(f'Taux arrestation  : {y2.mean():.1%}')
print(f'scale_pos_weight  : {scale_pos_weight:.2f}')


In [ ]:
if RUN_HYPERPARAM_SEARCH:
    param_dist = {
        'n_estimators': [200, 300, 400],
        'max_depth': [4, 5, 6, 7],
        'learning_rate': [0.03, 0.05, 0.08, 0.1],
        'subsample': [0.75, 0.85, 1.0],
        'colsample_bytree': [0.75, 0.85, 1.0],
        'min_child_weight': [1, 3, 5],
        'gamma': [0, 0.1, 0.2],
    }
    xgb_base = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0,
        n_jobs=-1,
    )
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        xgb_base,
        param_distributions=param_dist,
        n_iter=20,
        scoring='roc_auc',
        cv=cv,
        random_state=RANDOM_STATE,
        verbose=1,
        n_jobs=-1,
    )
    sample_idx = X2_train.sample(min(30_000, len(X2_train)), random_state=RANDOM_STATE).index
    search.fit(X2_train.loc[sample_idx], y2_train.loc[sample_idx])
    xgb_params = search.best_params_
    print('Meilleurs hyperparametres :')
    print(xgb_params)
else:
    xgb_params = {
        'n_estimators': 300,
        'max_depth': 5,
        'learning_rate': 0.05,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'min_child_weight': 3,
        'gamma': 0.1,
    }

xgb_arrest = XGBClassifier(
    **xgb_params,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1,
)
xgb_arrest.fit(X2_train, y2_train, eval_set=[(X2_test, y2_test)], verbose=False)

y2_pred = xgb_arrest.predict(X2_test)
y2_prob = xgb_arrest.predict_proba(X2_test)[:, 1]

acc_s2 = accuracy_score(y2_test, y2_pred)
f1_s2 = f1_score(y2_test, y2_pred, average='weighted')
auc_s2 = roc_auc_score(y2_test, y2_prob)

print('STEP 2 - XGBoost')
print(f'Accuracy : {acc_s2:.3f}')
print(f'F1 score : {f1_s2:.3f}')
print(f'ROC AUC  : {auc_s2:.3f}')
print()
print(classification_report(y2_test, y2_pred, target_names=['Pas arrestation', 'Arrestation']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm2 = confusion_matrix(y2_test, y2_pred)
sns.heatmap(
    cm2, annot=True, fmt=',', cmap='Reds', ax=axes[0],
    xticklabels=['Pas arrestation', 'Arrestation'],
    yticklabels=['Pas arrestation', 'Arrestation'],
)
axes[0].set_title('Matrice de confusion - arrestation', fontweight='bold')
axes[0].set_xlabel('Predit')
axes[0].set_ylabel('Reel')

importances_s2 = pd.Series(xgb_arrest.feature_importances_, index=FEAT_STEP2).sort_values()
axes[1].barh(importances_s2.index, importances_s2.values, color='#e15759', edgecolor='white')
axes[1].set_title('Feature importance - arrestation', fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

fpr, tpr, _ = roc_curve(y2_test, y2_prob)
axes[2].plot(fpr, tpr, color='#e15759', linewidth=2, label=f'AUC={auc_s2:.3f}')
axes[2].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Aleatoire')
axes[2].set_title('Courbe ROC', fontweight='bold')
axes[2].set_xlabel('False positive rate')
axes[2].set_ylabel('True positive rate')
axes[2].legend()
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(PRED_DIR / '04_step2_xgboost.png', bbox_inches='tight')
plt.show()


## 6. Fonction de prediction end-to-end


In [ ]:
def build_feature_row(latitude, longitude, hour, day_of_week, month, domestic, location_group):
    is_weekend = int(day_of_week >= 5)
    coord_scaled = scaler_kmeans.transform(pd.DataFrame([{
        'latitude': latitude,
        'longitude': longitude,
    }]))
    cluster = int(kmeans.predict(coord_scaled)[0])

    x_step1 = pd.DataFrame([{
        'latitude': latitude,
        'longitude': longitude,
        'hour': hour,
        'day_of_week': day_of_week,
        'month': month,
        'is_weekend': is_weekend,
        'cluster': cluster,
        'location_group': location_group,
    }])[FEAT_STEP1]

    crime_encoded = int(rf_crime.predict(x_step1)[0])
    crime_label = le_crime.inverse_transform([crime_encoded])[0]
    crime_confidence = float(rf_crime.predict_proba(x_step1)[0].max())

    x_step2 = pd.DataFrame([{
        'latitude': latitude,
        'longitude': longitude,
        'hour': hour,
        'day_of_week': day_of_week,
        'month': month,
        'is_weekend': is_weekend,
        'domestic': int(domestic),
        'cluster': cluster,
        'location_group': location_group,
        'crime_predicted': crime_encoded,
    }])[FEAT_STEP2]

    arrest_probability = float(xgb_arrest.predict_proba(x_step2)[0, 1])
    arrest_prediction = int(arrest_probability >= 0.50)

    return {
        'cluster': cluster,
        'crime_predicted': crime_label,
        'crime_confidence': crime_confidence,
        'arrest_prediction': arrest_prediction,
        'arrest_probability': arrest_probability,
    }


In [ ]:
examples = pd.DataFrame([
    {
        'scenario': 'Residence, mardi 14h, domestic',
        'latitude': 41.875,
        'longitude': -87.720,
        'hour': 14,
        'day_of_week': 1,
        'month': 6,
        'domestic': 1,
        'location_group': LOCATION_GROUP_MAPPING['Residence'],
    },
    {
        'scenario': 'Rue, samedi 3h, downtown',
        'latitude': 41.882,
        'longitude': -87.628,
        'hour': 3,
        'day_of_week': 5,
        'month': 11,
        'domestic': 0,
        'location_group': LOCATION_GROUP_MAPPING['Street'],
    },
    {
        'scenario': 'Espace public, mercredi 15h',
        'latitude': 41.870,
        'longitude': -87.715,
        'hour': 15,
        'day_of_week': 2,
        'month': 4,
        'domestic': 0,
        'location_group': LOCATION_GROUP_MAPPING['Public'],
    },
])

predictions = []
for row in examples.to_dict('records'):
    result = build_feature_row(
        latitude=row['latitude'],
        longitude=row['longitude'],
        hour=row['hour'],
        day_of_week=row['day_of_week'],
        month=row['month'],
        domestic=row['domestic'],
        location_group=row['location_group'],
    )
    predictions.append({**row, **result})

predictions_df = pd.DataFrame(predictions)
predictions_df['crime_confidence'] = predictions_df['crime_confidence'].map(lambda value: f'{value:.1%}')
predictions_df['arrest_probability'] = predictions_df['arrest_probability'].map(lambda value: f'{value:.1%}')
predictions_df


## 7. Sauvegarde des artefacts


In [ ]:
metadata = {
    'random_state': RANDOM_STATE,
    'start_year': START_YEAR,
    'api_limit': API_LIMIT,
    'n_clusters': N_CLUSTERS,
    'features_step1': FEAT_STEP1,
    'features_step2': FEAT_STEP2,
    'location_group_mapping': LOCATION_GROUP_MAPPING,
    'classes_crime': le_crime.classes_.tolist(),
    'metrics': {
        'step1_accuracy': float(acc_s1),
        'step1_f1_weighted': float(f1_s1),
        'step2_accuracy': float(acc_s2),
        'step2_f1_weighted': float(f1_s2),
        'step2_roc_auc': float(auc_s2),
    },
}

joblib.dump(rf_crime, MODEL_DIR / 'rf_crime_type.pkl')
joblib.dump(xgb_arrest, MODEL_DIR / 'xgb_arrest.pkl')
joblib.dump(kmeans, MODEL_DIR / 'kmeans_chicago.pkl')
joblib.dump(scaler_kmeans, MODEL_DIR / 'scaler_kmeans.pkl')
joblib.dump(le_crime, MODEL_DIR / 'label_encoder_crime.pkl')
joblib.dump(metadata, MODEL_DIR / 'metadata.joblib')

with open(MODEL_DIR / 'metadata.json', 'w') as file:
    json.dump(metadata, file, indent=2)

print('Artefacts sauvegardes :')
for path in sorted(MODEL_DIR.iterdir()):
    print(f'- {path.relative_to(PROJECT_ROOT)}')
